In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

In [ ]:
import torch

text = open('input.txt', encoding='utf-8').read()
chars = sorted(set(text))                      # 문자 단위 vocab (약 65개)
stoi = {c: i for i, c in enumerate(chars)}
itos = {i: c for c, i in stoi.items()}
encode = lambda s: [stoi[c] for c in s]
decode = lambda ids: ''.join(itos[i] for i in ids)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.9 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split, B=16, T=64):
    d = train_data if split == 'train' else val_data
    ix = torch.randint(len(d) - T - 1, (B,))          # 랜덤 시작점 B개
    x = torch.stack([d[i:i+T] for i in ix])           # 입력  (B, T)
    y = torch.stack([d[i+1:i+T+1] for i in ix])       # 정답  (B, T), 한 칸 밀림
    return x, y

In [ ]:
x, y = get_batch('train')
print(x.shape, y.shape)          # torch.Size([16, 64]) 둘 다
print(repr(decode(x[0, :20].tolist())))
print(repr(decode(y[0, :20].tolist())))   # 한 글자씩 밀려 있어야 함

In [ ]:
import math
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadAttention(nn.Module):
    def __init__(self, d, h, max_T=256):
        super().__init__()
        assert d % h == 0
        self.h, self.dk = h, d // h
        self.qkv = nn.Linear(d, 3 * d, bias=False)   # W_Q, W_K, W_V를 한 행렬로
        self.wo = nn.Linear(d, d, bias=False)        # W_O
        mask = torch.tril(torch.ones(max_T, max_T, dtype=torch.bool))
        self.register_buffer('mask', mask)           # 학습 안 되는 상수

    def forward(self, x, mask):
        B, T, d = x.shape
        q, k, v = self.qkv(x).split(d, dim=-1)                          # 각 (B, T, d)
        q, k, v = [t.view(B, T, self.h, self.dk).transpose(1, 2)
                   for t in (q, k, v)]                                   # (B, h, T, dk)
        scores = q @ k.transpose(-2, -1) / math.sqrt(self.dk)           # (B, h, T, T)
        scores = scores.masked_fill(~mask.to(x.device), float('-inf'))  # 미래 칸 막기
        w = F.softmax(scores, dim=-1)
        self.last_w = w.detach()
        out = (w @ v).transpose(1, 2).reshape(B, T, d)                  # head concat
        return self.wo(out)

In [ ]:
class Block(nn.Module):
    def __init__(self, d, h, max_T):
        super().__init__()
        self.ln1 = nn.LayerNorm(d)
        self.attn = MultiHeadAttention(d, h, max_T)
        self.ln2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, 4*d), nn.GELU(), nn.Linear(4*d, d))

    def forward(self, x, mask):
        x = x + self.attn(self.ln1(x), mask)     # 소통
        x = x + self.ffn(self.ln2(x))      # 처리
        return x


class TinyGPT(nn.Module):
    def __init__(self, V, d=128, h=4, n_layer=4, max_T=256):
        super().__init__()
        self.max_T = max_T
        self.tok_emb = nn.Embedding(V, d)
        self.pos_emb = nn.Embedding(max_T, d)           # 학습형 위치 embedding
        self.blocks = nn.ModuleList([Block(d, h, max_T) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, V, bias=False)
        self.head.weight = self.tok_emb.weight          # weight tying
        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.zeros_(m.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        pos = torch.arange(T, device=idx.device)
        x = self.tok_emb(idx) + self.pos_emb(pos)       # (B,T,d) + (T,d)
        for blk in self.blocks:
            x = blk(x)
        logits = self.head(self.ln_f(x))                # (B, T, V)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(B*T, -1), targets.view(B*T))
        return logits, loss

In [ ]:
def causal_mask(T):
    return torch.tril(torch.ones(T, T, dtype=torch.bool))

def bidirectional_mask(T):
    return torch.ones(T, T, dtype=torch.bool)

def prefix_lm_mask(T, n_prefix):
    m = causal_mask(T)
    m[:n_prefix, :n_prefix] = True      # prefix 블록만 양방향으로 풀어 줌
    return m

print(prefix_lm_mask(4, 2).int())       # 6단계에서 그린 표와 같은지 확인

In [ ]:
import matplotlib.pyplot as plt

T = 8
masks = {'causal': causal_mask(T),
         'bidirectional': bidirectional_mask(T),
         'prefix-LM (4)': prefix_lm_mask(T, 4)}

torch.manual_seed(0)
attn = MultiHeadAttention(d=32, h=4)
x = torch.randn(1, T, 32)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, (name, m) in zip(axes, masks.items()):
    attn(x, m)
    ax.imshow(attn.last_w[0, 0])      # 배치 0, head 0의 (T, T) 가중치
    ax.set_title(name)
    ax.set_xlabel('key (주는 쪽)'); ax.set_ylabel('query (받는 쪽)')
plt.show()

In [ ]:
class PatchEmbed(nn.Module):
    def __init__(self, img=32, patch=4, in_ch=3, d=128):
        super().__init__()
        self.n = (img // patch) ** 2
        self.proj = nn.Conv2d(in_ch, d, kernel_size=patch, stride=patch)

    def forward(self, img):                       # (B, 3, 32, 32)
        x = self.proj(img)                        # (B, d, 8, 8)
        return x.flatten(2).transpose(1, 2)       # (B, 64, d): 패치 64개 시퀀스

In [ ]:
pe = PatchEmbed()
img = torch.randn(1, 3, 32, 32)
out = pe(img)                                              # (1, 64, 128)

patch0 = img[0, :, 0:4, 0:4].reshape(-1)                   # 첫 패치: 숫자 48개
manual = pe.proj.weight.reshape(128, -1) @ patch0 + pe.proj.bias
print(torch.allclose(out[0, 0], manual, atol=1e-5))        # True

In [ ]:
class TinyViT(nn.Module):
    def __init__(self, n_cls=10, img=32, patch=4, d=128, h=4, n_layer=6):
        super().__init__()
        self.patch = PatchEmbed(img, patch, 3, d)
        N = self.patch.n
        self.cls = nn.Parameter(torch.zeros(1, 1, d))               # [CLS] 벡터
        self.pos = nn.Parameter(torch.randn(1, N + 1, d) * 0.02)    # 학습형 위치
        self.blocks = nn.ModuleList([Block(d, h, N + 1) for _ in range(n_layer)])
        self.ln_f = nn.LayerNorm(d)
        self.head = nn.Linear(d, n_cls)
        self.register_buffer('mask', bidirectional_mask(N + 1))

    def forward(self, img):
        x = self.patch(img)                                # (B, N, d)
        cls = self.cls.expand(x.shape[0], -1, -1)          # (B, 1, d)
        x = torch.cat([cls, x], dim=1) + self.pos          # (B, N+1, d)
        for blk in self.blocks:
            x = blk(x, self.mask)
        return self.head(self.ln_f(x[:, 0]))               # CLS 위치만 → (B, 10)

In [ ]:
import torchvision, torchvision.transforms as Tf

# tf = Tf.Compose([Tf.ToTensor(), Tf.Normalize((0.5,)*3, (0.5,)*3)])
# train_ds = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=tf)
# test_ds  = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=tf)
# train_dl = torch.utils.data.DataLoader(train_ds, batch_size=128, shuffle=True)
# test_dl  = torch.utils.data.DataLoader(test_ds, batch_size=256)

# vit = TinyViT().to(device)
# opt = torch.optim.AdamW(vit.parameters(), lr=3e-4)

# for epoch in range(10):
#     vit.train()
#     for img, label in train_dl:
#         loss = F.cross_entropy(vit(img.to(device)), label.to(device))
#         opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
#     vit.eval(); correct = 0
#     with torch.no_grad():
#         for img, label in test_dl:
#             correct += (vit(img.to(device)).argmax(-1).cpu() == label).sum().item()
#     print(epoch, loss.item(), correct / len(test_ds))

In [ ]:
def clip_loss(img_emb, txt_emb, t):
    img = F.normalize(img_emb, dim=-1)
    txt = F.normalize(txt_emb, dim=-1)
    logits = img @ txt.T * t                                   # (N, N) 유사도 × 온도
    labels = torch.arange(len(img), device=img.device)         # 정답 = 대각선
    return (F.cross_entropy(logits, labels) +                  # 행: 이미지 → 캡션
            F.cross_entropy(logits.T, labels)) / 2             # 열: 캡션 → 이미지

def siglip_loss(img_emb, txt_emb, t, b):
    img = F.normalize(img_emb, dim=-1)
    txt = F.normalize(txt_emb, dim=-1)
    logits = img @ txt.T * t + b                               # (N, N), bias 추가
    labels = 2 * torch.eye(len(img), device=img.device) - 1    # 대각 +1, 나머지 −1
    return -F.logsigmoid(labels * logits).sum() / len(img)     # 칸마다 독립

In [ ]:
torch.manual_seed(0)
D_img, D_txt, D = 64, 48, 32
R = torch.randn(D_img, D_txt)                     # 숨겨진 이미지-캡션 대응 규칙

def make_pairs(N, noise=8.0):
    x = torch.randn(N, D_img)
    y = x @ R + noise * torch.randn(N, D_txt)     # 짝 캡션 = 변환 + 노이즈
    return x, y

def train(loss_name, N, steps=2000):
    f_img, f_txt = nn.Linear(D_img, D), nn.Linear(D_txt, D)   # 두 "인코더"
    log_t = nn.Parameter(torch.tensor(math.log(10.0)))        # 온도 (log로 학습)
    b = nn.Parameter(torch.tensor(-10.0))                     # SigLIP bias
    params = [*f_img.parameters(), *f_txt.parameters(), log_t, b]
    opt = torch.optim.Adam(params, lr=1e-3)

    for _ in range(steps):
        x, y = make_pairs(N)
        ei, et = f_img(x), f_txt(y)
        if loss_name == 'clip':
            loss = clip_loss(ei, et, log_t.exp())
        else:
            loss = siglip_loss(ei, et, log_t.exp(), b)
        opt.zero_grad(); loss.backward(); opt.step()

    x, y = make_pairs(1000)                       # 평가: 1000개 중 짝 찾기
    with torch.no_grad():
        sim = F.normalize(f_img(x), dim=-1) @ F.normalize(f_txt(y), dim=-1).T
    acc = (sim.argmax(dim=1) == torch.arange(1000)).float().mean().item()
    return acc, log_t.exp().item(), b.item()

for name in ['clip', 'siglip']:
    for N in [4, 16, 64, 256]:
        acc, t, bias = train(name, N)
        print(f'{name:7s} N={N:4d}  acc={acc:.3f}  t={t:6.2f}  b={bias:6.2f}')